# Merging observations from two or more time series

## Overview

IrisPie provides four methods for merging observations from multiple time series:

* `overlay_by_span()` — Overlay values from another series over the entire span (including missing values)
* `overlay_by_observation()` — Overlay only valid (non-missing) observations from another series
* `underlay_by_span()` — Underlay values from another series over the entire span
* `underlay_by_observation()` — Underlay only valid observations from another series

All methods are available both as **instance methods** (modifying the series in place) and as **functions** (returning a new series without modifying the original).


## Setup

Import the IrisPie package and create two time series, `x` and `y`, with the following characteristics:

* **Overlapping but distinct spans**: `x` goes from `2020-Q1` to `2021-Q4`, `y` goes from `2020-Q4` to `2022-Q3`
* **In-sample missing observations**: `x` has a missing value in `2021-Q3`, `y` has a missing value in `2021-Q1`

These characteristics help illustrate the differences between the merge methods.


In [ ]:
import irispie as ir

x = ir.Series(start=ir.qq(2020,1), values=(1, 2, 3, 4, 5, 6, None, 8))
y = ir.Series(start=ir.qq(2020,4), values=(10, None, 30, 40, 50, 60, 70, 80))

print("Original series x and y:")
print(x | y)

## Method 1: `overlay_by_span()`

The `overlay_by_span()` method overlays all observations from another series over the entire span of that series, **including missing values**. This means:

1. The resulting series spans from the earliest start to the latest end of both series
2. Within the span of `y`, all values from `y` (including `NaN`) overwrite values from `x`
3. Outside the span of `y`, values from `x` are preserved

**Key behavior**: The missing value in `y` at `2021-Q1` overwrites the value `5` from `x`.


In [ ]:
# Using as a method (modifies in place)
xx = x.copy()
xx.overlay_by_span(y)

print("Result of overlay_by_span | original x | original y:")
print(xx | x | y)

## Method 2: `overlay_by_observation()`

The `overlay_by_observation()` method overlays **only valid (non-missing) observations** from another series. This means:

1. The resulting series spans from the earliest start to the latest end of both series
2. Only where `y` has valid (non-`NaN`) values are those values superimposed on `x`
3. Where `y` has missing values, the original values from `x` are preserved

**Key behavior**: The value `5` in `x` at `2021-Q1` is preserved because `y` has a missing value there.


In [ ]:
# Using as a method (modifies in place)
xx = x.copy()
xx.overlay_by_observation(y)

print("Result of overlay_by_observation | original x | original y:")
print(xx | x | y)

## Method 3: `underlay_by_span()`

The `underlay_by_span()` method places values from another series **beneath** the current series over the entire span. This is the inverse of `overlay_by_span()`. The logic is:

1. The resulting series spans from the earliest start to the latest end of both series
2. Values from `y` are used as the base
3. All values from `x` (including missing values) within its span overwrite values from `y`
4. Outside the span of `x`, values from `y` are used

**Key behavior**: The missing value in `x` at `2021-Q3` overwrites the value `40` from `y`.


In [ ]:
# Using as a method (modifies in place)
xx = x.copy()
xx.underlay_by_span(y)

print("Result of underlay_by_span | original x | original y:")
print(xx | x | y)

## Method 4: `underlay_by_observation()`

The `underlay_by_observation()` method fills in missing observations in the current series with valid values from another series. This is the inverse of `overlay_by_observation()`. The logic is:

1. The resulting series spans from the earliest start to the latest end of both series
2. Values from `y` are used as the base
3. Only where `x` has valid (non-`NaN`) values are those values superimposed on `y`
4. Where `x` has missing values, the values from `y` are preserved

**Key behavior**: The missing value in `x` at `2021-Q3` is filled with value `40` from `y`. This is the most common use case for "filling gaps" in data.


In [ ]:
# Using as a method (modifies in place)
xx = x.copy()
xx.underlay_by_observation(y)

print("Result of underlay_by_observation | original x | original y:")
print(xx | x | y)

## Using functional forms

All four methods are also available as **functions** (not just methods). Functions:

* Do **not** modify the original series
* Return a **new series** with the merged data
* Are useful in functional programming style or when you want to avoid modifying data in place

The functional forms are:
* `ir.overlay_by_span(series1, series2)`
* `ir.overlay_by_observation(series1, series2)`
* `ir.underlay_by_span(series1, series2)`
* `ir.underlay_by_observation(series1, series2)`


In [ ]:
# overlay_by_span as a function
z1 = ir.overlay_by_span(x, y)

print("overlay_by_span (functional):")
print(z1 | x | y)

In [ ]:
# overlay_by_observation as a function
z2 = ir.overlay_by_observation(x, y)

print("overlay_by_observation (functional):")
print(z2 | x | y)

In [ ]:
# underlay_by_span as a function
z3 = ir.underlay_by_span(x, y)

print("underlay_by_span (functional):")
print(z3 | x | y)

In [ ]:
# underlay_by_observation as a function
z4 = ir.underlay_by_observation(x, y)

print("underlay_by_observation (functional):")
print(z4 | x | y)

## Quick comparison

Here's a side-by-side comparison of all four methods applied to the same data:


In [ ]:
# Compare all methods
result_overlay_span = ir.overlay_by_span(x, y)
result_overlay_obs = ir.overlay_by_observation(x, y)
result_underlay_span = ir.underlay_by_span(x, y)
result_underlay_obs = ir.underlay_by_observation(x, y)

print("Comparison of all four methods:")
print(result_overlay_span | result_overlay_obs | result_underlay_span | result_underlay_obs)

## Summary

| Method | Behavior | Use case |
| --- | --- | --- |
| `overlay_by_span()` | Overwrites entire span (including NaN) | Replace data completely over a time range |
| `overlay_by_observation()` | Overwrites only valid observations | Update data while preserving values where new data is missing |
| `underlay_by_span()` | Places base data under entire span | Provide base data that gets overwritten completely |
| `underlay_by_observation()` | Fills gaps with valid observations | **Most common**: Fill missing values with fallback data |

**When to use each:**

* Use `overlay_by_observation()` when you want to **update** data with new values but keep existing values where new data is unavailable
* Use `underlay_by_observation()` when you want to **fill gaps** in your primary data with backup/fallback values
* Use `*_by_span()` variants when you need to handle missing values explicitly, including cases where missing values should overwrite existing data
